In [ ]:
# This is a simple model for predicting heart disease, using data from https://archive.ics.uci.edu/dataset/45/heart+disease.
# The model's performance isn't well using MSE loss for a binary classification task. Adjustments will be made later.
# After some experiments, the model has best accuracy of 85.25%

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import lightning as L
from torch.utils.data import TensorDataset, DataLoader

import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
url1 = "./heart_disease_output/train.txt"
url2 = "./heart_disease_output/test.txt"
df1 = pd.read_table(url1)
df2 = pd.read_table(url2)

In [ ]:
x_train = df1.drop(columns="label")

In [ ]:
y_train = df1["label"]

In [ ]:
x_test = df2.drop(columns="label")

In [ ]:
y_test = df2["label"]

In [ ]:
one_hot_label_train = F.one_hot(torch.tensor(y_train)).type(torch.float32)

In [ ]:
max_vals_in_x_train = x_train.max()

In [ ]:
min_vals_in_x_train = x_train.min()

In [ ]:
train_medians = x_train.median()

x_train = x_train.fillna(train_medians)
x_test = x_test.fillna(train_medians)

In [ ]:
x_train = (x_train - min_vals_in_x_train) / (max_vals_in_x_train - min_vals_in_x_train)

In [ ]:
x_test = (x_test - min_vals_in_x_train) / (max_vals_in_x_train - min_vals_in_x_train)

In [ ]:
x_train_tensors = torch.tensor(x_train.values).type(torch.float32)

In [ ]:
x_test_tensors = torch.tensor(x_test.values).type(torch.float32)

In [ ]:
assert not x_train.isna().any().any()
assert not x_test.isna().any().any()

In [ ]:
train_dataset = TensorDataset(x_train_tensors, one_hot_label_train)
train_dataloader = DataLoader(train_dataset)

In [ ]:
class heart_disease(L.LightningModule):
    def __init__(self):
        super().__init__()
        L.seed_everything(42)
        self.input_to_hidden = nn.Linear(in_features=13, out_features=4, bias=True)
        self.hidden_to_output = nn.Linear(in_features=4, out_features=2, bias=True)
        self.loss = nn.MSELoss(reduction='sum')

    def forward(self, input):
        hidden = self.input_to_hidden(input)
        output_values = self.hidden_to_output(torch.relu(hidden))
        return output_values

    def configure_optimizers(self):
        return Adam(self.parameters(), lr = 0.001)

    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self.forward(inputs)
        loss = self.loss(outputs, labels)
        return loss
    

In [ ]:
model = heart_disease()

In [ ]:
trainer = L.Trainer(max_epochs=100)
torch.autograd.set_detect_anomaly(True)
trainer.fit(model, train_dataloaders=train_dataloader)

In [ ]:
predictions = model(x_test_tensors)
predicted_labels = torch.argmax(predictions, dim=1)
torch.sum(torch.eq(torch.tensor(y_test), predicted_labels)) / len(predicted_labels)